# AISdb Complete Walkthrough

This walkthrough is the definitive end-to-end tour of AISdb. It runs the full
practitioner pipeline against the bundled `example_data.db` sample, from opening a
database connection through querying, building vessel tracks, cleaning, interpolation,
speed and distance calculations, CSV export, and a final interactive map. Every step
uses the sample database and package test data only, so the notebook executes top to
bottom without any external service.

The sample covers 54 vessels in the Gulf of Maine between 2022-01-01 and 2022-01-20 UTC.
Every query window in this walkthrough stays inside that range.

### What you will learn

- How to connect to an AISdb SQLite database and (optionally) PostgreSQL
- How to query AIS data by time range, bounding box, and MMSI with the right callbacks
- How to turn query rows into vessel tracks with `TrackGen`
- How to clean noisy tracks by segmenting on time gaps and re-linking with the encoder
- How to interpolate a track to a regular time step
- How to compute per-segment distance (`delta_meters`) and speed (`delta_knots`)
- How to export cleaned tracks to CSV with `write_csv`
- How to visualize tracks on an interactive map with the built-in web interface

The narrative companion for this walkthrough is the whole AISdb book. Start from the
[introduction](https://aisviz.gitbook.io/documentation/introduction).

## Setup

In [ ]:
# Install AISdb if it is not already available (uncomment to run).
# On Google Colab, run this cell first, then restart the runtime before continuing.
# %pip install aisdb

We import the public API from the top-level `aisdb` package. The callback functions
that filter a query live under `aisdb.database.sqlfcn_callbacks`, and the connection
classes come from `aisdb.database.dbconn`. Only the standard scientific stack is needed
beyond AISdb itself.

In [ ]:
import numpy as np
from datetime import datetime, timedelta

import aisdb
from aisdb import SQLiteDBConn, DBQuery, DomainFromPoints
from aisdb.database import sqlfcn_callbacks as cb

The bundled `example_data.db` sits next to this notebook, so we point `dbpath` at it
directly. Two flags gate the parts of the notebook that reach outside the sample data.
`RUN_POSTGRES` stays `False` so the PostgreSQL example never executes against placeholder
credentials, and `SHOW_MAP` stays `False` so the final map cell does not try to open a
browser during headless execution. Flip either to `True` when you run locally against a
real database or a desktop session.

In [ ]:
dbpath = './example_data.db'   # bundled sample database (SQLite)

RUN_POSTGRES = False   # set True locally to run the PostgreSQL variant below
SHOW_MAP = False       # set True locally to open the interactive map at the end

# The sample spans 2022-01-01 to 2022-01-20 UTC. A one-week window is a good default.
start_time = datetime(2022, 1, 1)
end_time = datetime(2022, 1, 8)

## Connect to the database

AISdb opens a database through a connection object used as a context manager. For the
sample we use `SQLiteDBConn`. The connection is a lazy stream: `DBQuery.gen_qry()` and
`TrackGen` read from it as they are consumed, so all querying and processing must happen
inside the `with` block while the connection is open. We confirm the connection by
counting the vessels present in the sample.

In [ ]:
with SQLiteDBConn(dbpath=dbpath) as dbconn:
    cur = dbconn.cursor()
    cur.execute("SELECT COUNT(DISTINCT mmsi) FROM ais_202201_dynamic")
    n_vessels = cur.fetchone()[0]

print(f"Sample database holds {n_vessels} distinct vessels.")

### PostgreSQL variant (optional)

AISdb also speaks PostgreSQL, optionally with the TimescaleDB extension, for larger and
shared deployments. The API is identical once connected; only the connection object
changes. `PostgresDBConn` takes the same keyword arguments as `psycopg`, and
`hostaddr=...` accepts a raw IP address. The cell below is guarded by `RUN_POSTGRES` so it
is skipped during headless runs and never executes against placeholder credentials. Fill
in real values and set `RUN_POSTGRES = True` to use it. Never pass both `dbconn` and
`dbpath` to `DBQuery`; give it a connection object only.

In [ ]:
if RUN_POSTGRES:
    from aisdb.database.dbconn import PostgresDBConn

    with PostgresDBConn(
        hostaddr='127.0.0.1',   # PostgreSQL host IP
        port=5432,              # PostgreSQL port
        user='USERNAME',        # PostgreSQL user
        password='PASSWORD',    # PostgreSQL password
        dbname='DBNAME',        # database name
    ) as pg_conn:
        qry = DBQuery(
            dbconn=pg_conn, start=start_time, end=end_time,
            callback=cb.in_timerange_validmmsi,
        )
        pg_tracks = list(aisdb.track_gen.TrackGen(qry.gen_qry(), decimate=False))
        print(f"PostgreSQL returned {len(pg_tracks)} tracks.")
else:
    print("PostgreSQL example skipped (set RUN_POSTGRES = True to run it).")

## Query and build tracks

Querying has two parts. `DBQuery` describes what to fetch (a time range, an optional
bounding box, an optional MMSI, and a callback that filters rows). `TrackGen` turns the
returned rows into tracks, where each track is a dictionary holding one vessel's
time-ordered arrays of `lon`, `lat`, `sog`, and the other dynamic fields.

The callback is what selects a query type. Here `in_timerange_validmmsi` fetches every
valid-MMSI vessel active in the window. We materialize the generator into a list so we can
reuse the tracks across later cells.

In [ ]:
with SQLiteDBConn(dbpath=dbpath) as dbconn:
    qry = DBQuery(
        dbconn=dbconn, start=start_time, end=end_time,
        callback=cb.in_timerange_validmmsi,
    )
    tracks = list(aisdb.track_gen.TrackGen(qry.gen_qry(), decimate=False))

print(f"Retrieved {len(tracks)} vessel tracks for the week.")
sample = tracks[0]
print(f"First track: MMSI {sample['mmsi']} with {sample['lon'].size} positions.")

### Query within a bounding box

To restrict a query in space, define a domain and pass its boundary to `DBQuery`, then
switch the callback to one that filters on the box. `DomainFromPoints` builds a bounding
box extending a given radius (in meters) around a point. Here we take a 50 km box around a
point off the Gulf of Maine coast and use `in_time_bbox_validmmsi` to combine the time and
space filters.

In [ ]:
domain = DomainFromPoints(points=[(-68.5, 43.5)], radial_distances=[50000])

with SQLiteDBConn(dbpath=dbpath) as dbconn:
    qry = DBQuery(
        dbconn=dbconn, start=start_time, end=end_time,
        xmin=domain.boundary['xmin'], xmax=domain.boundary['xmax'],
        ymin=domain.boundary['ymin'], ymax=domain.boundary['ymax'],
        callback=cb.in_time_bbox_validmmsi,
    )
    bbox_tracks = list(aisdb.track_gen.TrackGen(qry.gen_qry(), decimate=False))

print(f"{len(bbox_tracks)} tracks fall inside the 50 km box during the week.")

### Query a single vessel by MMSI

Passing an `mmsi` and the `in_timerange_hasmmsi` callback pulls the full history of one
vessel over the whole sample window. We pick a busy vessel here and keep its single track
for the cleaning, interpolation, and calculation sections that follow, so the rest of the
notebook tells one continuous story about one ship.

In [ ]:
VESSEL_MMSI = 368194280   # a busy vessel present across the whole sample

with SQLiteDBConn(dbpath=dbpath) as dbconn:
    qry = DBQuery(
        dbconn=dbconn,
        start=datetime(2022, 1, 1), end=datetime(2022, 1, 20),
        mmsi=VESSEL_MMSI,
        callback=cb.in_timerange_hasmmsi,
    )
    vessel_tracks = list(aisdb.track_gen.TrackGen(qry.gen_qry(), decimate=False))

vessel = vessel_tracks[0]
print(f"Vessel {vessel['mmsi']} has {vessel['lon'].size} raw positions over the sample.")

## Clean the tracks

Raw AIS is noisy. A single MMSI can carry position jumps from receiver glitches, duplicate
identifiers, or spoofing, and long idle gaps blur separate voyages together. AISdb cleans
this in two chained steps that each accept and return a track generator.

`split_timedelta` cuts a track wherever the gap between consecutive pings exceeds a
threshold, so distinct voyages become distinct segments. `encode_greatcircledistance` then
splits any segment where consecutive pings imply an impossible speed or distance, and
re-links the fragments that plausibly belong to the same voyage using a score. Because the
inputs are lazy generators, we rebuild the query inside the `with` block and consume the
whole chain there.

In [ ]:
with SQLiteDBConn(dbpath=dbpath) as dbconn:
    qry = DBQuery(
        dbconn=dbconn, start=start_time, end=end_time,
        callback=cb.in_timerange_validmmsi,
    )
    raw = aisdb.track_gen.TrackGen(qry.gen_qry(), decimate=False)

    # Step 1: split on time gaps longer than 24 hours.
    segments = aisdb.split_timedelta(raw, timedelta(hours=24))

    # Step 2: split on implausible jumps, then re-link plausible segments.
    cleaned = aisdb.encode_greatcircledistance(
        segments,
        distance_threshold=20000,   # meters
        speed_threshold=50,         # knots
        minscore=1e-6,
    )
    clean_tracks = list(cleaned)

print(f"Cleaning produced {len(clean_tracks)} coherent track segments from the week.")

## Interpolate a track

Interpolation fills the gaps between reported positions with estimated points at a regular
step, which is useful when a downstream analysis needs evenly spaced samples. AISdb offers
several methods; linear time interpolation with `interp_time` is the fast default. We apply
it to our single vessel at a 10-minute step. Interpolation invents positions between real
reports, so clean a track before interpolating rather than after.

In [ ]:
with SQLiteDBConn(dbpath=dbpath) as dbconn:
    qry = DBQuery(
        dbconn=dbconn,
        start=datetime(2022, 1, 1), end=datetime(2022, 1, 20),
        mmsi=VESSEL_MMSI,
        callback=cb.in_timerange_hasmmsi,
    )
    raw = aisdb.track_gen.TrackGen(qry.gen_qry(), decimate=False)
    interpolated = list(aisdb.interp.interp_time(raw, timedelta(minutes=10)))

interp_track = interpolated[0]
print(f"Raw positions:          {vessel['lon'].size}")
print(f"Interpolated (10 min):  {interp_track['lon'].size}")

## Distance and speed calculations

Two `aisdb.gis` helpers turn a track's positions into per-segment metrics. `delta_meters`
returns the haversine distance in meters between each pair of consecutive positions, and
`delta_knots` returns the implied speed over ground in knots for each of those gaps. Both
return an array of length one less than the number of positions. `delta_knots` clamps each
elapsed time to at least one second internally, so duplicate timestamps cannot blow up the
speed. We run both on the raw single-vessel track.

In [ ]:
dist_m = aisdb.gis.delta_meters(vessel)
speed_kn = aisdb.gis.delta_knots(vessel)

print(f"Segments: {len(dist_m)}")
print(f"Distance (m):  min {dist_m.min():.1f}   mean {dist_m.mean():.1f}   max {dist_m.max():.1f}")
print(f"Speed (knots): min {np.min(speed_kn):.2f}   mean {np.mean(speed_kn):.2f}   max {np.max(speed_kn):.2f}")

A very high maximum speed is the exact signal the cleaning step removes. The threshold we
passed to `encode_greatcircledistance` above (50 knots) flags jumps like these as noise and
splits the track there, which is why the cleaned segments hold physically plausible motion.

## Export to CSV

`aisdb.write_csv` takes a track generator and writes one row per AIS message, with a
`Track_ID` column so segments stay groupable after loading the file elsewhere. It handles
column ordering and value sanitization. The query, track generation, and write all happen
inside the same `with` block because the generators read from the open connection as
`write_csv` consumes them. We export the cleaned tracks for the week.

In [ ]:
output_csv = './walkthrough_tracks.csv'

with SQLiteDBConn(dbpath=dbpath) as dbconn:
    qry = DBQuery(
        dbconn=dbconn, start=start_time, end=end_time,
        callback=cb.in_timerange_validmmsi,
    )
    raw = aisdb.track_gen.TrackGen(qry.gen_qry(), decimate=False)
    segments = aisdb.split_timedelta(raw, timedelta(hours=24))
    cleaned = aisdb.encode_greatcircledistance(
        segments, distance_threshold=20000, speed_threshold=50, minscore=1e-6,
    )
    aisdb.write_csv(cleaned, output_csv)

import os
print(f"Wrote {os.path.getsize(output_csv) / 1024:.1f} KB to {output_csv}")

## Visualize on an interactive map

`aisdb.web_interface.visualize` renders tracks on an interactive map. It starts a small
local web server and opens a browser, so it cannot run headless. The call is guarded by the
`SHOW_MAP` flag we set at the top, and we import `nest_asyncio` only here, where the web
interface actually needs it. Set `SHOW_MAP = True` in a local desktop session to see the
cleaned tracks on the map.

In [ ]:
if SHOW_MAP:
    import nest_asyncio
    nest_asyncio.apply()

    def color_tracks(track_iter, color='yellow'):
        for track in track_iter:
            track['color'] = color
            yield track

    with SQLiteDBConn(dbpath=dbpath) as dbconn:
        qry = DBQuery(
            dbconn=dbconn, start=start_time, end=end_time,
            callback=cb.in_timerange_validmmsi,
        )
        raw = aisdb.track_gen.TrackGen(qry.gen_qry(), decimate=False)
        segments = aisdb.split_timedelta(raw, timedelta(hours=24))
        cleaned = aisdb.encode_greatcircledistance(
            segments, distance_threshold=20000, speed_threshold=50, minscore=1e-6,
        )
        aisdb.web_interface.visualize(
            color_tracks(cleaned),
            visualearth=False,
            open_browser=True,
        )
else:
    print("Map skipped (set SHOW_MAP = True locally to open the interactive map).")

## Next steps

This walkthrough touched every stage of the AISdb pipeline once. Each stage has a focused
tutorial that goes deeper, and the numbered notebooks in the
[AISViz Tutorials repository](https://github.com/MAPS-Lab/AISdb-Tutorials) follow the same order:
database loading, querying, cleaning, visualization, interpolation, calculations, using
your own AIS data, CSV export, and bathymetric enrichment.

For the full narrative, environmental-context helpers (shore, coast, port, and bathymetry
distances), and machine-learning examples, work through the
[AISdb documentation](https://aisviz.gitbook.io/documentation/introduction) end to end.